# Weather-Sensitive Zone Demand

**Grain:** pickup year/month × pickup zone × weather category. Weather codes follow `analytical_queries.ipynb`. The denominator is the number of observed citywide weather hours in that month/category, including hours with zero trips in a particular zone. Thus `avg_hourly_demand` is comparable across weather categories with different exposure times. The baseline uses only classified, matched weather hours; missing/unknown conditions are excluded. This describes association, not causation.


In [1]:
from pathlib import Path
import sys

# Find the project even when Jupyter starts in its notebooks directory.
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root / "notebooks" / "data_products"))
from product_utils import get_spark, register_integrated, publish_product

spark = get_spark()
register_integrated(spark)


D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Build the product

The SQL below defines the grain and metrics. Run this cell again to refresh the Delta table.


In [2]:
# Spark SQL keeps the grouping logic visible and reusable.
query = """
WITH classified_hours AS (
    SELECT DISTINCT
        weather_observation_timestamp AS weather_hour,
        YEAR(weather_observation_timestamp) AS pickup_year,
        MONTH(weather_observation_timestamp) AS pickup_month,
        CASE
            WHEN weather_condition_code IN (1, 2) THEN 'clear_or_fair'
            WHEN weather_condition_code IN (3, 4) THEN 'cloudy'
            WHEN weather_condition_code IN (5, 6) THEN 'fog'
            WHEN weather_condition_code IN (7, 8, 9, 10, 11, 17, 18) THEN 'rain'
            WHEN weather_condition_code IN (12, 13, 14, 15, 16, 19, 20, 21, 22) THEN 'snow_or_sleet'
            WHEN weather_condition_code BETWEEN 23 AND 27 THEN 'storm'
            ELSE 'unknown'
        END AS weather_condition
    FROM integrated_taxi_trips
    WHERE weather_match_status = 'exact_hour'
      AND weather_observation_timestamp IS NOT NULL
), valid_hours AS (
    SELECT * FROM classified_hours WHERE weather_condition <> 'unknown'
), zones AS (
    SELECT DISTINCT pickup_location_id,
           COALESCE(pickup_zone, 'Unknown') AS pickup_zone,
           COALESCE(pickup_borough, 'Unknown') AS pickup_borough
    FROM integrated_taxi_trips
    WHERE pickup_location_id IS NOT NULL
), zone_hour_trips AS (
    SELECT pickup_location_id, pickup_hour, COUNT(trip_id) AS trip_count
    FROM integrated_taxi_trips
    WHERE trip_id IS NOT NULL
      AND weather_match_status = 'exact_hour'
      AND pickup_location_id IS NOT NULL
    GROUP BY pickup_location_id, pickup_hour
), zone_weather_hours AS (
    -- The cross join adds zero-trip hours for every observed zone.
    SELECT z.pickup_location_id, z.pickup_zone, z.pickup_borough,
           h.pickup_year, h.pickup_month, h.weather_condition,
           COALESCE(t.trip_count, 0) AS trip_count
    FROM zones z
    CROSS JOIN valid_hours h
    LEFT JOIN zone_hour_trips t
      ON z.pickup_location_id = t.pickup_location_id
     AND h.weather_hour = t.pickup_hour
), category_summary AS (
    SELECT pickup_year, pickup_month, pickup_location_id,
           pickup_zone, pickup_borough, weather_condition,
           COUNT(*) AS weather_hour_count,
           SUM(trip_count) AS trip_count,
           AVG(trip_count) AS avg_hourly_demand
    FROM zone_weather_hours
    GROUP BY pickup_year, pickup_month, pickup_location_id,
             pickup_zone, pickup_borough, weather_condition
), with_baseline AS (
    SELECT *,
           SUM(trip_count) OVER (
               PARTITION BY pickup_year, pickup_month, pickup_location_id
           ) / SUM(weather_hour_count) OVER (
               PARTITION BY pickup_year, pickup_month, pickup_location_id
           ) AS zone_month_avg_hourly_demand
    FROM category_summary
)
SELECT pickup_year, pickup_month, pickup_location_id,
       pickup_zone, pickup_borough, weather_condition,
       weather_hour_count, trip_count,
       ROUND(avg_hourly_demand, 3) AS avg_hourly_demand,
       ROUND(zone_month_avg_hourly_demand, 3) AS zone_month_avg_hourly_demand,
       ROUND(100.0 * (avg_hourly_demand - zone_month_avg_hourly_demand)
             / NULLIF(zone_month_avg_hourly_demand, 0), 2)
             AS demand_vs_zone_month_pct
FROM with_baseline
"""
result = spark.sql(query)
product = publish_product(spark, result, "weather_sensitive_zone_demand", ())
product.printSchema()
product.show(20, truncate=False)


Saved weather_sensitive_zone_demand to D:\Master\data intensive\data-intensive-computing\data\processed\analytical_products\weather_sensitive_zone_demand
root
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- weather_hour_count: long (nullable = true)
 |-- trip_count: long (nullable = true)
 |-- avg_hourly_demand: double (nullable = true)
 |-- zone_month_avg_hourly_demand: double (nullable = true)
 |-- demand_vs_zone_month_pct: double (nullable = true)
 |-- source_table: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- refreshed_at: timestamp (nullable = true)
 |-- schema_version: string (nullable = true)

+-----------+------------+------------------+-----------------------+--------------+-----------------+------------------+------